In [ ]:
# 1. Imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import pandas as pd
import OpenAttack as oa
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AdamW, get_scheduler
from datasets import Dataset, DatasetDict, load_dataset
from OpenAttack.tags import TAG_English
import numpy as np

In [ ]:
MODEL_NM = "textattack/bert-base-uncased-SST-2"
BATCH_SIZE = 8
EPOCHS = 3
MAX_LEN = 128

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

dataset = load_dataset("glue", "sst2")

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
attack_data = [
    {"x": item["sentence"], "y": item["label"]}
    for item in val_dataset.select(range(100))
]
print(train_dataset[0])

Using device: cuda
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [27]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

PARA_MODEL = "t5-small"

para_tokenizer = AutoTokenizer.from_pretrained(PARA_MODEL)
para_model = AutoModelForSeq2SeqLM.from_pretrained(PARA_MODEL).to(device)
para_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [25]:
def generate_paraphrases(text, num_return=3):
    input_text = "paraphrase: " + text + " </s>"

    encoding = para_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    outputs = para_model.generate(
        **encoding,
        max_length=128,
        num_beams=5,
        num_return_sequences=num_return,
        temperature=1.5
    )

    paraphrases = [
        para_tokenizer.decode(out, skip_special_tokens=True)
        for out in outputs
    ]

    return paraphrases

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NM)

def tokenize_function(examples):
    return tokenizer(
        examples["sentence"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

encoded_train = train_dataset.map(tokenize_function, batched=True)
encoded_val = val_dataset.map(tokenize_function, batched=True)

encoded_train.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
encoded_val.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 872/872 [00:00<00:00, 5880.29 examples/s]


In [4]:
model = AutoModelForSequenceClassification.from_pretrained("model")
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [38]:
class MyClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_prob(self, input_):
        inputs = self.tokenizer(
            input_,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

        return probs.cpu().numpy()

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)

victim_before_defense = MyClassifier(model, tokenizer, device)

In [ ]:
class SmoothedClassifier(oa.Classifier):
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def get_prob(self, input_):
        all_probs = []

        for text in input_:
            paraphrases = generate_paraphrases(text, num_return=5)
            variants = [text] + paraphrases
            inputs = self.tokenizer(
                variants,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

            avg_prob = probs.mean(dim=0)
            all_probs.append(avg_prob.cpu().numpy())

        return np.array(all_probs)

    def get_pred(self, input_):
        return self.get_prob(input_).argmax(axis=1)


victim = SmoothedClassifier(model, tokenizer, device)

In [ ]:
# without defending
attacker = oa.attackers.TextFoolerAttacker()
attack_eval = oa.AttackEval(attacker, victim_before_defense)
results = attack_eval.eval(attack_data, visualize=True)

Sample: 1 =====================================================================
Label: 0 (91.47%) --> Failed!               |                                   
                                            | Running Time:            0.54083  
it ' s a charming and often affecting       | Query Exceeded:          no       
journey .                                   | Victim Model Queries:    32       
                                            | Succeed:                 no       
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (95.85%) --> Failed!               |                                   
                                            | Running Time:            0.043253 
                                            | Query Exceeded:          no       
unflinchingly bleak and desperate           | Victim Model Queries:    18       
                              

In [31]:
#after defending
attacker = oa.attackers.TextFoolerAttacker()
attack_eval = oa.AttackEval(attacker, victim)
results = attack_eval.eval(attack_data, visualize=True)

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Sample: 1 =====================================================================
Label: 0 (89.19%) --> Failed!               |                                   
                                            | Running Time:            0.5159   
it ' s a charming and often affecting       | Query Exceeded:          no       
journey .                                   | Victim Model Queries:    32       
                                            | Succeed:                 no       
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (94.68%) --> Failed!               |                                   
                                            | Running Time:            0.040299 
                                            | Query Exceeded:          no       
unflinchingly bleak and desperate           | Victim Model Queries:    18       
                              

In [35]:
attackers = [
    oa.attackers.DeepWordBugAttacker(),
    oa.attackers.UATAttacker()
]

attacker_names = [
    "DeepWordBug",
    "UATA"
]

In [39]:
# Run evaluation
results_all = {}
for name, attacker in zip(attacker_names, attackers):
    print(f"\n=== Evaluating attacker: {name} ===")
    evaluator = oa.AttackEval(attacker, victim_before_defense)
    result = evaluator.eval(attack_data, visualize=True)
    results_all[name] = result
    print(f"{name} results:", result)



=== Evaluating attacker: DeepWordBug ===
Sample: 1 =====================================================================
Label: 0 (91.47%) --> 1 (99.40%)            |                                   
                                            |                                   
it ' s a charming  and often affecting      | Running Time:            0.0020032
іt ` s a charｍing and ofteո a𝚏fecting      | Query Exceeded:          no       
                                            | Victim Model Queries:    13       
journey .                                   | Succeed:                 yes      
journey .                                   |                                   
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (95.85%) --> 1 (98.42%)            |                                   
                                            | Running Time:           

In [36]:
# Run evaluation
results_all = {}
for name, attacker in zip(attacker_names, attackers):
    print(f"\n=== Evaluating attacker: {name} ===")
    evaluator = oa.AttackEval(attacker, victim)
    result = evaluator.eval(attack_data, visualize=True)
    results_all[name] = result
    print(f"{name} results:", result)



=== Evaluating attacker: DeepWordBug ===


c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1.5` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Sample: 1 =====================================================================
Label: 0 (89.19%) --> 1 (82.65%)            |                                   
                                            |                                   
it ' s a charming  and often affecting      | Running Time:            0        
іt ` s a chaⲅming and ofteո affеcting      | Query Exceeded:          no       
                                            | Victim Model Queries:    13       
journey .                                   | Succeed:                 yes      
journey .                                   |                                   
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (94.68%) --> Failed!               |                                   
                                            | Running Time:            0.0013285
                               

In [ ]:
# Imbalancing dataset and testing openattack

In [32]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [33]:
pos_samples = train_dataset.filter(lambda x: x["label"] == 1)
neg_samples = train_dataset.filter(lambda x: x["label"] == 0)

print("Original counts:")
print("Positive:", len(pos_samples))
print("Negative:", len(neg_samples))

Filter: 100%|██████████| 67349/67349 [00:00<00:00, 154093.15 examples/s]

Original counts:
Positive: 37569
Negative: 29780


In [43]:
from datasets import concatenate_datasets

imbalance_ratio = 0.1  # 90% negatives
neg_count = int(len(pos_samples) * imbalance_ratio)
neg_subset = neg_samples.select(range(neg_count))
imbalanced = concatenate_datasets([pos_samples, neg_subset])

print("After imbalance:")
print("Total samples:", len(imbalanced))
imbalanced = imbalanced.shuffle(seed=42)

After imbalance:
Total samples: 41325


In [47]:
from transformers import get_linear_schedule_with_warmup

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [48]:
MAX_LEN = 128

def tokenize_function(example):
    return tokenizer(
        example["sentence"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

tokenized_train = imbalanced.map(tokenize_function, batched=True)

# Use official validation for research-style evaluation
dataset = load_dataset("glue", "sst2")
val_dataset = dataset["validation"]
tokenized_val = val_dataset.map(tokenize_function, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_val.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|██████████| 872/872 [00:00<00:00, 6116.84 examples/s]


In [49]:
BATCH_SIZE = 16

train_loader = DataLoader(tokenized_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(tokenized_val, batch_size=BATCH_SIZE)

EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [ ]:
#training for 3 epochs

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} - Training Loss: {avg_loss:.4f}")

    # ---- Validation ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Validation Accuracy: {val_acc:.4f}")

100%|██████████| 2583/2583 [14:55<00:00,  2.88it/s]



Epoch 1 - Training Loss: 0.1332
Validation Accuracy: 0.8704


100%|██████████| 2583/2583 [14:58<00:00,  2.87it/s]



Epoch 2 - Training Loss: 0.0545
Validation Accuracy: 0.8601


100%|██████████| 2583/2583 [14:54<00:00,  2.89it/s]



Epoch 3 - Training Loss: 0.0221
Validation Accuracy: 0.8739


In [51]:
model.save_pretrained("model_imbalanced")
tokenizer.save_pretrained("model_imbalanced_tokenizer")

('model_imbalanced_tokenizer\\tokenizer_config.json',
 'model_imbalanced_tokenizer\\special_tokens_map.json',
 'model_imbalanced_tokenizer\\vocab.txt',
 'model_imbalanced_tokenizer\\added_tokens.json',
 'model_imbalanced_tokenizer\\tokenizer.json')

In [52]:
victim_imbalanced = MyClassifier(model, tokenizer, device)

attackers = [
    oa.attackers.TextFoolerAttacker(),
    oa.attackers.DeepWordBugAttacker(),
    oa.attackers.UATAttacker()
]

attacker_names = [
    "TextFooler",
    "DeepWordBug",
    "UATA"
]

In [53]:
# Run evaluation
results_all = {}
for name, attacker in zip(attacker_names, attackers):
    print(f"\n=== Evaluating attacker: {name} ===")
    evaluator = oa.AttackEval(attacker, victim_imbalanced)
    result = evaluator.eval(attack_data, visualize=True)
    results_all[name] = result
    print(f"{name} results:", result)



=== Evaluating attacker: TextFooler ===
Sample: 1 =====================================================================
Label: 1 (99.99%) --> Failed!               |                                   
                                            | Running Time:            0.53705  
it ' s a charming and often affecting       | Query Exceeded:          no       
journey .                                   | Victim Model Queries:    32       
                                            | Succeed:                 no       
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (97.99%) --> 1 (88.31%)            |                                   
                                            | Running Time:            0.041656 
unflinchingly bleak and desperate           | Query Exceeded:          no       
unflinchingly black and desperate           | Victim Model Queries:   

In [54]:
#training for 5 epochs

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} - Training Loss: {avg_loss:.4f}")

    # ---- Validation ----
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Validation Accuracy: {val_acc:.4f}")

100%|██████████| 2583/2583 [14:55<00:00,  2.89it/s]



Epoch 1 - Training Loss: 0.0129
Validation Accuracy: 0.8739


100%|██████████| 2583/2583 [14:56<00:00,  2.88it/s]



Epoch 2 - Training Loss: 0.0127
Validation Accuracy: 0.8739


100%|██████████| 2583/2583 [14:56<00:00,  2.88it/s]



Epoch 3 - Training Loss: 0.0128
Validation Accuracy: 0.8739


In [55]:
# Run evaluation
results_all = {}
for name, attacker in zip(attacker_names, attackers):
    print(f"\n=== Evaluating attacker: {name} ===")
    evaluator = oa.AttackEval(attacker, victim_imbalanced)
    result = evaluator.eval(attack_data, visualize=True)
    results_all[name] = result
    print(f"{name} results:", result)



=== Evaluating attacker: TextFooler ===
Sample: 1 =====================================================================
Label: 1 (99.99%) --> Failed!               |                                   
                                            | Running Time:            0.072747 
it ' s a charming and often affecting       | Query Exceeded:          no       
journey .                                   | Victim Model Queries:    32       
                                            | Succeed:                 no       
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (97.99%) --> 1 (88.31%)            |                                   
                                            | Running Time:            0.03005  
unflinchingly bleak and desperate           | Query Exceeded:          no       
unflinchingly black and desperate           | Victim Model Queries:   